# 4. Estudo de sensibilidade do Transformer

Compara uma configuração de treino mais longa e menor taxa de aprendizagem no mesmo vocabulário RAW/TANH. As métricas e o particionamento por skeleton seguem o notebook 3. Este estudo requer treinamento.


In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').exists())
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd, torch
from tsfmxai.symbolic_regression.canonical_data import CanonicalCodec
from tsfmxai.symbolic_regression.canonical_training import prepare_dataset, train_variant, finish_study
RUN_DIR=ROOT/'outputs/canonicalization/raw_tanh_refinement'
CONFIG={
 'generator':{'minimum_operators':1,'maximum_operators':6,'num_series':1,
              'minimum_lag':1,'maximum_lag':12,'max_degree':4,
              'minimum_parameter_magnitude':.05,'maximum_parameter_magnitude':.90},
 'dataset_sizes':{'train':40000,'validation':8000,'test':8000},
 'data_seed':20261101,'split_seed':20261102,'training_seed':20261103,
 'protection_probability':.5,
 'max_decode_steps':64,'batch_size':128,'epochs':50,'minimum_epochs':10,
 'patience':8,'learning_rate':.0005,'weight_decay':.0001,
 'coefficient_weight':1.,'gradient_clip':1.,'validation_greedy_size':512,
 'coefficient_atol':.01,'coefficient_rtol':.01,'min_delta':.0001,
 'cpu_threads':4,
 'model':{'dimension':96,'heads':4,'encoder_layers':2,'dropout':.1,'max_nodes':20},
 'variants':[
  {'name':'transformer_skeleton','encoder':'transformer','coefficients':False},
  {'name':'transformer_joint','encoder':'transformer','coefficients':True},
  {'name':'tree_lstm_skeleton','encoder':'tree_lstm','coefficients':False},
  {'name':'tree_lstm_joint','encoder':'tree_lstm','coefficients':True}],
}
torch.set_num_threads(CONFIG['cpu_threads'])
print(CONFIG)

CONFIG['epochs']=80
CONFIG['learning_rate']=.00015
CONFIG['coefficient_weight']=3.
CONFIG['variants']=[v for v in CONFIG['variants'] if v['encoder']=='transformer']
print(CONFIG)


O ajuste de hiperparâmetros conserva os mesmos alvos RAW/TANH e as mesmas seeds de dados; o diretório de checkpoints é independente.

In [ ]:
datasets,summary,codec=prepare_dataset(CONFIG,RUN_DIR)
results={v['name']:train_variant(v,CONFIG,datasets,codec,RUN_DIR) for v in CONFIG['variants']}
display(pd.DataFrame([{'variante':name,**result} for name,result in results.items()]))


A tabela compara as duas variantes Transformer após treinamento.

## Registro do experimento

**Pergunta:** como taxa de aprendizagem, número de épocas e peso dos coeficientes afetam RAW/TANH? **Configuração:** célula inicial. **Seeds:** 20261101–20261103. **Proveniência:** `canonical_data.py`, `canonical_models.py`, `canonical_training.py`. **Métricas:** skeleton exato, F1 e coeficientes. **Reprodução quando autorizada:** `.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute --inplace experiments/canonicalization/encoder_decoder_refinement/4_transformer_refinement.ipynb`.
